# Long-Term Memory Prioritization Demo (SQLite)

This notebook is the notebook-converted version of `demo/08_itemized_insights.py`.

It demonstrates bounded long-term memory behavior inspired by human memory:
- **Recency**: newer insights receive higher short-term retention
- **Frequency**: repeatedly cited insights become stronger
- **Forgetting**: old unused insights decay and are pruned
- **Bounded Capacity**: only top-N insights are retained

## Requirements
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_KEY`
- Optional `AZURE_OPENAI_EMB_DEPLOYMENT`, `AZURE_OPENAI_PROCESSING_MODEL`

In [ ]:
import asyncio
import os
import sys
from pathlib import Path
from datetime import datetime
from typing import Dict, List

from dotenv import load_dotenv

# Resolve project root whether launched from workspace root or demo folder.
cwd = Path.cwd().resolve()
BASE_DIR = cwd.parent if cwd.name == 'demo' else cwd
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# Force .env values to override inherited kernel environment values.
load_dotenv(BASE_DIR / '.env', override=True)

from openai import AzureOpenAI
from memory.db.sqlite_backend import SQLiteDatabase
from memory.db.base import ContainerType
from memory.providers.embedding import OpenAIEmbeddingProvider
from memory.core.reflection import Reflection, ReflectionConfig
from memory.core.insight_items import (
    LongTermInsightItem,
    InsightIdGenerator,
    rank_insights,
    calculate_retention_score,
    SessionAnalysisWithCitations,
    build_context_with_ids,
)

USER_ID = 'memory_priority_demo'
DB_PATH = str(BASE_DIR / 'demo_memory_priority.db')
MAX_INSIGHTS = 5

client = AzureOpenAI(
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_key=os.environ.get('AZURE_OPENAI_API_KEY'),
    api_version='2024-10-21',
)

TIMELINE = [
    {
        'session_id': 'month-1',
        'simulated_date': datetime(2025, 1, 15),
        'title': 'Month 1: Initial Consultation',
        'summary': 'Alex is 28, software engineer earning $120k. Wants to start investing. Very risk-averse due to parents 2008 losses. Prefers bonds and savings. Has $10,000 emergency fund. Interested in Roth IRA.',
        'turns': [
            ('user', "I'm Alex, 28, making $120k. I want to start investing but I'm scared of losing money."),
            ('assistant', "That's understandable. Let's start with your risk tolerance and goals."),
            ('user', "My parents lost a lot in 2008. I want safe investments - bonds, savings accounts."),
            ('assistant', "With your fear of losses, we can start conservatively. You mentioned a Roth IRA?"),
            ('user', "Yes, I want to learn about Roth IRAs. I have $10k saved for emergencies."),
        ],
    },
    {
        'session_id': 'month-2',
        'simulated_date': datetime(2025, 2, 20),
        'title': 'Month 2: Roth IRA Setup',
        'summary': 'Alex opened a Roth IRA last month and asks about contribution limits. Still conservative with money market allocation. Mentions saving for sisters wedding.',
        'turns': [
            ('user', "I opened the Roth IRA! How much can I contribute this year?"),
            ('assistant', "Great! The 2025 limit is $7,000. Given your conservative preference, what did you choose?"),
            ('user', "I put it in a money market fund. I know it's low return but I can't handle volatility."),
            ('assistant', "That's fine to start. Any other financial goals coming up?"),
            ('user', "My sister's wedding is in April. I need to save about $2,000 for travel and gift."),
        ],
    },
    {
        'session_id': 'month-3',
        'simulated_date': datetime(2025, 3, 25),
        'title': 'Month 3: Tax Season Questions',
        'summary': 'Alex asks tax implications of Roth contributions and plans to max out the Roth IRA. Risk profile still conservative.',
        'turns': [
            ('user', "Quick tax question - do my Roth IRA contributions reduce my taxable income?"),
            ('assistant', "No, Roth contributions are post-tax. But withdrawals in retirement are tax-free."),
            ('user', "I think I want to max out my Roth this year."),
            ('assistant', "Good plan. Still comfortable with money market holdings?"),
            ('user', "Yes, not ready for stocks yet. Maybe next year."),
        ],
    },
    {
        'session_id': 'month-4',
        'simulated_date': datetime(2025, 5, 10),
        'title': 'Month 4: Post-Wedding, Promotion News',
        'summary': 'Wedding complete. Alex promoted to $150k and starts reconsidering risk tolerance. Asks about target-date funds.',
        'turns': [
            ('user', "Big news! Wedding was great, and I just got promoted to $150k!"),
            ('assistant', "Congratulations! How are you feeling about finances now?"),
            ('user', "More confident. Maybe I can handle some risk."),
            ('assistant', "Would you like to explore middle-ground options?"),
            ('user', "Yes, target-date funds seem balanced."),
        ],
    },
    {
        'session_id': 'month-5',
        'simulated_date': datetime(2025, 6, 15),
        'title': 'Month 5: Risk Tolerance Shift',
        'summary': 'Alex moves Roth IRA to target-date 2060 fund and starts taxable account planning for home down payment.',
        'turns': [
            ('user', "I moved my Roth to a 2060 target-date fund."),
            ('assistant', "Great step. How do you feel about it?"),
            ('user', "Good. Now I want a taxable account too."),
            ('assistant', "Any goal for the taxable account?"),
            ('user', "House down payment in 3-4 years."),
        ],
    },
    {
        'session_id': 'month-6',
        'simulated_date': datetime(2025, 7, 20),
        'title': 'Month 6: Current State & Planning',
        'summary': 'Alex keeps house fund conservative while becoming aggressive for long-term retirement and considers higher 401k contributions.',
        'turns': [
            ('user', "I started the house fund with $5,000 and kept it conservative."),
            ('assistant', "Good time-horizon matching. How is your 401k?"),
            ('user', "Should I increase 401k contributions now?"),
            ('assistant', "At $150k, maxing 401k is ideal if cash flow allows."),
            ('user', "I'll do that."),
        ],
    },
]

print(f'Base directory: {BASE_DIR}')
print(f'SQLite DB path: {DB_PATH}')
print('Setup complete.')

In [ ]:
def print_section(title: str, char: str = '=') -> None:
    width = 70
    print(f'\n{char * width}')
    print(f' {title}')
    print(f'{char * width}')


def print_insight_table(items: List[LongTermInsightItem], now: datetime, title: str = 'Current Insights') -> None:
    print(f'\n{title}:')
    print('-' * 95)
    print(f"{'ID':<10} {'Score':>6} {'Access':>7} {'Age':>8} {'Importance':>10}  {'Text':<40}")
    print('-' * 95)

    ranked = rank_insights(items, now)
    for item, score in ranked:
        age_days = (now - item.date_added).days
        age_str = f'{age_days}d' if age_days < 30 else f'{age_days // 30}m {age_days % 30}d'
        text = item.insight_text[:38] + '..' if len(item.insight_text) > 38 else item.insight_text
        print(f"{item.id:<10} {score:>6.2f} {item.access_count:>7} {age_str:>8} {item.importance:>10}  {text}")
    print('-' * 95)


async def get_insight_items(db: SQLiteDatabase, user_id: str) -> List[LongTermInsightItem]:
    items = await db.query(
        container=ContainerType.INSIGHTS,
        filters={'user_id': user_id, 'insight_type': 'long_term_item'},
        order_by='-created_at',
    )

    result = []
    for item_data in items:
        try:
            result.append(LongTermInsightItem.from_dict(item_data))
        except Exception as exc:
            print(f'  Warning: Could not parse insight item: {exc}')
    return result


async def prune_insights(
    db: SQLiteDatabase,
    user_id: str,
    items: List[LongTermInsightItem],
    max_items: int,
    now: datetime,
) -> tuple[list[LongTermInsightItem], list[LongTermInsightItem]]:
    if len(items) <= max_items:
        return items, []

    ranked = rank_insights(items, now)
    kept = [item for item, _ in ranked[:max_items]]
    pruned = [item for item, _ in ranked[max_items:]]

    for item in pruned:
        await db.delete(container=ContainerType.INSIGHTS, document_id=item.id, partition_key=user_id)

    return kept, pruned


async def run_session_with_simulated_time(
    reflection: Reflection,
    db: SQLiteDatabase,
    emb_provider: OpenAIEmbeddingProvider,
    user_id: str,
    session: Dict,
    simulated_now: datetime,
) -> Dict:
    from memory.prompts import SESSION_ANALYSIS_WITH_CITATIONS_PROMPT

    existing_items = await get_insight_items(db, user_id)
    existing_context = build_context_with_ids(existing_items)

    turns_text = '\n'.join([f"{role}: {content}" for role, content in session['turns']])
    full_context = f"{session['summary']}\n\nConversation:\n{turns_text}"

    prompt = SESSION_ANALYSIS_WITH_CITATIONS_PROMPT.format(
        existing_insights_context=existing_context,
        session_content=full_context,
    )

    try:
        analysis = reflection._call_llm_with_json(
            system_prompt='You are an expert session analysis assistant with memory tracking.',
            user_prompt=prompt,
            output_model=SessionAnalysisWithCitations,
        )
    except Exception as exc:
        print(f'  Error: {exc}')
        return {
            'session_summary': 'Error',
            'key_topics': [],
            'new_insights': [],
            'cited_insight_ids': [],
            'has_meaningful_content': False,
        }

    cited_ids = []
    for citation in analysis.cited_insights:
        cited_ids.append(citation.insight_id)
        doc = await db.get_by_id(
            container=ContainerType.INSIGHTS,
            document_id=citation.insight_id,
            partition_key=user_id,
        )
        if doc and doc.get('insight_type') == 'long_term_item':
            doc['access_count'] = doc.get('access_count', 0) + 1
            doc['last_accessed'] = simulated_now.isoformat()
            await db.upsert(container=ContainerType.INSIGHTS, document=doc, partition_key=user_id)

    id_gen = InsightIdGenerator([item.id for item in existing_items])
    new_insight_items = []

    for insight in analysis.new_insights:
        item = LongTermInsightItem(
            id=id_gen.next_id(),
            user_id=user_id,
            agent_id='default',
            insight_text=insight.insight_text,
            category=insight.category,
            confidence=insight.confidence,
            importance=insight.importance,
            date_added=simulated_now,
            last_accessed=simulated_now,
            access_count=0,
            source_session_ids=[session['session_id']],
        )
        new_insight_items.append(item)

    for item in new_insight_items:
        item.embedding = emb_provider.get_embedding(item.insight_text)
        await db.upsert(container=ContainerType.INSIGHTS, document=item.to_dict(), partition_key=user_id)

    return {
        'session_summary': analysis.session_summary,
        'key_topics': analysis.key_topics,
        'new_insights': [i.to_dict() for i in new_insight_items],
        'cited_insight_ids': cited_ids,
        'has_meaningful_content': analysis.has_meaningful_content,
    }


print('Helper functions are ready.')

## Cell 4: Clear Execution Walkthrough

Cell 5 runs the full simulation pipeline and prints how long-term memory evolves over time under hard capacity constraints.

### What Cell 5 does
1. **Reset state**
- Deletes any old SQLite demo database so the run starts clean and reproducible.

2. **Initialize storage and model components**
- Creates `SQLiteDatabase` for local persistence.
- Creates embedding provider and reflection component for insight extraction + citation tracking.

3. **Simulate six monthly sessions**
- Each session contains user+assistant turns plus a high-level summary.
- Existing insights are injected into context with stable IDs.

4. **Extract and reinforce memory**
- New insights are generated and stored with simulated timestamps.
- Cited existing insights get `access_count += 1` and refreshed `last_accessed`.

5. **Apply bounded-memory pruning**
- If count exceeds `MAX_INSIGHTS=5`, items are ranked by retention score.
- Lower-score items are deleted, demonstrating forgetting behavior.

6. **Summarize evolution and cleanup**
- Prints month-by-month retention leaders and scores.
- Removes demo DB at end to avoid residue.

### How to interpret the table
- **Score**: retention priority (higher is more likely to survive).
- **Access**: number of reinforcement hits from citations.
- **Age**: how old the insight is at that simulated month.
- **Importance**: model-generated salience label.

In [ ]:
async def run_demo() -> None:
    print_section('LONG-TERM MEMORY PRIORITIZATION DEMO')
    print(
        f"\nThis demo simulates 6 months of client sessions with a financial advisor.\n\n"
        f"Key concepts demonstrated:\n"
        f"- RECENCY: New insights start with a grace-period boost\n"
        f"- FREQUENCY: Cited insights gain strength (access_count increases)\n"
        f"- FORGETTING: Old, uncited insights decay over time\n"
        f"- BOUNDED MEMORY: Only {MAX_INSIGHTS} insights retained\n"
    )

    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)

    db = SQLiteDatabase(DB_PATH)
    await db.initialize()

    emb_provider = OpenAIEmbeddingProvider(
        openai_client=client,
        model=os.environ.get('AZURE_OPENAI_EMB_DEPLOYMENT', 'text-embedding-ada-002'),
    )

    reflection = Reflection(
        agent_id='default',
        database=db,
        embedding_provider=emb_provider,
        chat_client=client,
        config=ReflectionConfig(
            PROCESSING_MODEL=(
                os.environ.get('AZURE_OPENAI_PROCESSING_MODEL')
                or os.environ.get('AZURE_OPENAI_REASONING_MODEL')
                or 'gpt-4o-mini'
            )
        ),
    )

    insight_history = []

    try:
        for month_idx, session in enumerate(TIMELINE, 1):
            simulated_now = session['simulated_date']
            print_section(f"SESSION {month_idx}: {session['title']}", '=')
            print(f"Simulated Date: {simulated_now.strftime('%B %d, %Y')}")

            items_before = await get_insight_items(db, USER_ID)
            if items_before:
                print_insight_table(items_before, simulated_now, 'Memory State BEFORE Session')
            else:
                print('\n[No existing insights - this is the first session]')

            print('\n[Processing session...]')
            result = await run_session_with_simulated_time(
                reflection, db, emb_provider, USER_ID, session, simulated_now
            )

            print(f"\nSummary: {result['session_summary'][:100]}...")
            print(f"New insights: {len(result['new_insights'])}")
            for ins in result['new_insights']:
                print(f"  - [{ins['id']}] {ins['insight_text'][:50]}...")

            if result['cited_insight_ids']:
                print(f"Cited existing: {result['cited_insight_ids']}")

            items_after = await get_insight_items(db, USER_ID)
            if len(items_after) > MAX_INSIGHTS:
                print(f"\nCapacity exceeded ({len(items_after)} > {MAX_INSIGHTS}). Pruning...")
                kept, pruned = await prune_insights(db, USER_ID, items_after, MAX_INSIGHTS, simulated_now)

                print('Forgotten:')
                for item in pruned:
                    score = calculate_retention_score(item, simulated_now)
                    age = (simulated_now - item.date_added).days
                    print(f"  - [{item.id}] score={score:.2f} age={age}d access={item.access_count}: {item.insight_text[:35]}...")

                print('Retained:')
                for item in kept:
                    score = calculate_retention_score(item, simulated_now)
                    age = (simulated_now - item.date_added).days
                    print(f"  - [{item.id}] score={score:.2f} age={age}d access={item.access_count}: {item.insight_text[:35]}...")

                items_after = kept

            print_insight_table(items_after, simulated_now, 'Memory State AFTER Session (Top 5)')

            insight_history.append({
                'month': month_idx,
                'date': simulated_now,
                'insights': [
                    (
                        i.id,
                        i.insight_text[:25] + '...',
                        i.access_count,
                        calculate_retention_score(i, simulated_now),
                    )
                    for i in sorted(
                        items_after,
                        key=lambda x: calculate_retention_score(x, simulated_now),
                        reverse=True,
                    )
                ],
            })

        print_section('DEMO COMPLETE - MEMORY EVOLUTION SUMMARY', '=')
        print('\nHow insights evolved over 6 months:')
        print('-' * 80)
        for record in insight_history:
            print(f"\nMonth {record['month']} ({record['date'].strftime('%B %Y')}):")
            for insight_id, text, access, score in record['insights']:
                print(f"  [{insight_id}] score={score:.2f} access={access}: {text}")

    finally:
        await db.close()
        if os.path.exists(DB_PATH):
            os.remove(DB_PATH)
        print('\n[Cleaned up database]')


await run_demo()